# Soldani - Second task - Benchmark

In [7]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [8]:
import json
import pandas as pd

from openai import OpenAI
from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect, spurious_effect,
    natural_direct_effect, natural_indirect_effect,
)

client = OpenAI()

In [9]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

In [10]:
import time

def run_fairmind(config: dict) -> tuple[dict, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    effects = {
        "TV": total_variation(bn, target, config["protected"], x0, x1),
        "TE": total_effect(bn, target, config["protected"], x0, x1),
        "SE": spurious_effect(bn, target, config["protected"], x0),
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, elapsed

ground_truth, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind — elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

2026-06-30 16:14:45.156 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'BDeu'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'hours-per-week': 'N', 'education': 'C', 'T_income': 'C'}
2026-06-30 16:14:45.285 | DEBUG    | src.effects:total_variation:248 - Computing total variation for target=('T_income', '>50K'), private_baseline=Female, private_mod=Male
2026-06-30 16:14:45.312 | DEBUG    | src.effects:spurious_effect:195 - Computing spurious effect for target=('T_income', '>50K'), private_value=Female


FairMind — elapsed time: 0.0339s
  TV: 0.194470
  TE: 0.183161
  SE: -0.007296
  DE: 0.137049
  IE: -0.046112


In [11]:
def build_gpt_prompt(config: dict, n_rows: int = 300) -> str:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()
    sample = df.sample(n=min(n_rows, len(df)), random_state=42)
    csv_str = sample.to_csv(index=False)

    return f"""You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET ({len(sample)} rows):
{csv_str}

VARIABLE ROLES:
- X (protected): "{config['protected']}", x0="{config['x0']}", x1="{config['x1']}"
- Y (target):    "{config['target_col']}", target state="{config['target_val']}"
- W (mediators): {config['mediators']}
- Z (confounders): {config['confounders']}

IDENTIFICATION FORMULAE (use these exactly):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)
- SE = TV - TE
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)

Return ONLY a JSON object, no other text:
{{
  "TV": <float>,
  "TE": <float>,
  "SE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_gpt_prompt(CONFIG)
print(prompt[:600], "\n[...]")

You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET (300 rows):
S2_gender,hours-per-week,education,T_income
Female,40,HS-grad,<=50K
Male,40,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,40,HS-grad,<=50K
Female,30,Bachelors,<=50K
Female,40,HS-grad,<=50K
Male,45,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,50,HS-grad,<=50K
Female,40,Some-college,<=50K
Male,70,Some-college,<=50K
Male,30,Bachelors,<=50K
Male,40,Some-college,<=50K
Male,45,11th,<=50K
Male,40,Bachelors,<=50K
Male,60,Assoc-voc,<=50K
Femal 
[...]


In [ ]:
def call_gpt(prompt: str, model: str = "o3-mini") -> tuple[dict, dict, float]:
    start = time.perf_counter()
    response = client.responses.create(
        model=model,
        reasoning={"effort": "high"},
        input=[
            {
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}],
            }
        ],
    )
    elapsed = time.perf_counter() - start

    usage = {
        "input_tokens":     response.usage.input_tokens,
        "output_tokens":    response.usage.output_tokens,
        "reasoning_tokens": response.usage.output_tokens_details.reasoning_tokens,
        "total_tokens":     response.usage.total_tokens,
    }

    # JSON Parsing
    raw = response.output_text.strip()

    # Remove backticks
    raw = raw.replace("```json", "").replace("```", "").strip()
    effects = json.loads(raw)

    return effects, usage, elapsed

gpt_effects, gpt_usage, gpt_time = call_gpt(prompt)

print(f"GPT — time: {gpt_time:.4f}s")
print(f"Token: input={gpt_usage['input_tokens']}, "
      f"output={gpt_usage['output_tokens']}, "
      f"reasoning={gpt_usage['reasoning_tokens']}, "
      f"total={gpt_usage['total_tokens']}")
print(json.dumps(gpt_effects, indent=2))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


JSONDecodeError: Extra data: line 9 column 1 (char 85)

In [ ]:
def compute_discrepancies(ground_truth: dict, gpt_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        gpt = float(gpt_effects.get(effect, float("nan")))
        abs_err = abs(gt - gpt)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
            "fairmind":    round(gt,  6),
            "gpt":         round(gpt, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, gpt_effects)
print(discrepancies.to_string(index=False))

effect  fairmind     gpt  abs_error  rel_error_%
    TV  0.194470  0.1812   0.013270         6.82
    TE  0.183161  0.2110   0.027839        15.20
    SE -0.007296 -0.0298   0.022504       308.44
    DE  0.137049  0.1410   0.003951         2.88
    IE -0.046112  0.0700   0.116112       251.80


In [ ]:
def save_results(config, ground_truth, gpt_effects, discrepancies, usage, fairmind_time, gpt_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "gpt":           gpt_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "gpt_seconds":      round(gpt_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, gpt_effects, discrepancies, gpt_usage, fairmind_time, gpt_time)

Saved: benchmark_results/adult_20260621_192807.json
